## <img style="float: left; padding-right: 10px; width: 100px" src="https://raw.githubusercontent.com/sbildea/SEED/main/imagini/unitbv_mie.png"> <center>Știința Explorării și Exploatării Datelor - SEED1</center>


##    <center>Curs 3a: Învățare nesupervizată: PCA și clusterizare</center>
**Universitatea Transilvania**<br>
**SIIPA, Semestrul I**<br>
**Instructor:** Conf. dr. Teodor &Scedil;tefan B&icirc;ldea<br>
**Contact:** teodor.bildea@unitbv.ro<br>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sbildea/SEED/blob/main/cursuri/03-invatare-nesupervizata/curs-03a-pca-clusterizare.ipynb)

In [ ]:
# Configurare curs SEED: rulați această celulă prima.
# Colab: descarcă materialele cursului și instalează pachetele lipsă (aprox. un minut).
# Local: găsește singură rădăcina repository-ului, oricare ar fi folderul curent al kernelului.
import os, sys
MODUL = "03-invatare-nesupervizata"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/SEED"):
        !git clone -q --depth 1 https://github.com/sbildea/SEED.git /content/SEED
        !pip install -q -r /content/SEED/requirements.txt
    RADACINA = "/content/SEED"
else:
    RADACINA = os.path.abspath(".")
    while not os.path.isdir(os.path.join(RADACINA, "seed")) and os.path.dirname(RADACINA) != RADACINA:
        RADACINA = os.path.dirname(RADACINA)
    import importlib.util, socket
    lipsa = [p for p in ("pandas", "matplotlib", "seaborn", "sklearn") if importlib.util.find_spec(p) is None]
    if lipsa:
        print("Lipsesc pachetele", lipsa, "- în terminal: pip install -r requirements-local.txt")
    try:
        socket.create_connection(("raw.githubusercontent.com", 443), 2).close()
    except OSError:
        print("Fără internet: imaginile din text nu se văd. O singură dată, în terminal: python tools/imagini_offline.py")
os.chdir(os.path.join(RADACINA, "cursuri", MODUL))
sys.path.insert(0, RADACINA)

## Cum lucrăm azi: tu, agentul și notebook-ul

Acest curs se joacă în trei: **lecția** (acest notebook), **tu** și **agentul tău** (Gemini în Colab, un agent din VS Code sau din Antigravity). Agentul scrie codul; tu decizi ce se cere, de ce, și verifici dacă rezultatul are sens. Fiecare idee este o **scenă** cu aceiași pași:

| Pas | Ce faci |
|---|---|
| **Concept** | citești 2-3 propoziții |
| **Prezice** | spui (sau scrii) ce crezi că va ieși, *înainte* să rulezi |
| **Cere agentului** | formulezi cererea; apoi rulezi codul de referință și îl compari cu ce a scris agentul |
| **Verifică** | compari cu valorile de control |
| **Capcană** | vezi ce poate merge prost, chiar dacă codul rulează |
| **Pentru client** | spui rezultatul în limbaj de manager |

**Primer** (lipește-l agentului la începutul conversației; dacă ai pierdut conversația sau agentul a rămas fără tokeni, lipește primerul și *Jurnalul de sesiune* de la finalul notebook-ului unui alt agent și scrie „continuă de aici”):

```text
Rolul tău: ești partenerul meu de lucru la cursul SEED1 (analiză de date și învățare automată), modulul 3: învățare nesupervizată (PCA, clusterizare, reguli de asociere). Eu sunt student și nu știu să programez în Python; eu decid ce se face și de ce, tu scrii codul, iar eu verific rezultatul.

Datele cursului:
- Se încarcă DOAR cu `from seed.date import incarca`. `incarca("lab-set2")` întoarce un dicționar nume_tabel -> pandas DataFrame; un tabel se ia și după poziție (`incarca("lab-set2")[1]`) sau după nume (`incarca("lab-set2", "general_data")`).
- Nu folosi căi de fișiere și nu descărca date de pe internet.
- Folosește pandas, matplotlib/seaborn, scikit-learn și mlxtend. Cod scurt, comentarii în română, un pas pe rând.

Reguli pentru acest modul (le aplici chiar dacă nu le cer eu):
1. Înainte de PCA sau k-means: elimină coloana țintă, ID-urile și coloanele constante, apoi standardizează coloanele numerice.
2. Pune `random_state=42` și `n_init=10` la k-means; spune câte rânduri și coloane rămân după fiecare pas.
3. Nu inventa coloane: dacă nu ești sigur, afișează `df.columns`.
4. Când interpretezi grupuri, folosește mediile coloanelor originale, nu doar numărul grupului.
5. Spune-mi dacă ceva din rezultat îți pare ciudat sau slab (de exemplu silhouette mic), în loc să-l prezinți ca pe un succes.

Cum lucrăm:
- Implicit ești în MOD TUTORE: înainte de a-mi da codul, îmi pui o singură întrebare scurtă despre concept („ce crezi că se va întâmpla?”). Nu rezolvi temele în locul meu; mă ajuți să verific.
- La cererea mea poți trece în MOD EXECUTANT (execuți direct și spui pe scurt ce ai făcut) sau în MOD RECENZENT (cauți greșeli în ce îți arăt: scalare uitată, coloane de ID, scurgeri de informație).
- Când scriu „închide scena”, completezi Jurnalul de sesiune în formatul de mai jos, numai din ce am discutat. Dacă ceva nu s-a discutat, scrii „nu știu”.

Formatul Jurnalului de sesiune:
## Jurnal de sesiune
- Modul / scena la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):

Răspunde acum doar cu: rolul tău în două propoziții și cele 5 reguli, pe scurt.
```

## Problema de azi

Directorul de resurse umane al unei companii: *„Ce tipuri de angajați avem, ca să adaptăm programele de retenție?”*

Nu avem o coloană „tip de angajat”. Nu există un răspuns corect de învățat; trebuie să **găsim structură** în date. Aceasta este **învățarea nesupervizată** (fără țintă), spre deosebire de cea supervizată, în care modelul învață dintr-o țintă cunoscută (modulele 4-6).

Datele sunt tabelul `general_data` din `lab-set2` (HR Analytics: 4.410 angajați, 24 de coloane).

**Planul de azi (≈75 min):** scena 0 cererea vagă · 1 distanța și scalarea · 2 corelația · 3 PCA · 4 k-means · 5 segmentele și decizia · scena finală: ce îi spunem clientului. Definițiile complete sunt în secțiunea **Referință** de la sfârșit. Reguli de asociere și recomandări: `curs-03b`.

## Scena 0. Cererea vagă

**Concept.** Un agent execută ce îi ceri, nu ce ai vrut să ceri. O cerere fără concepte în ea produce un rezultat care *arată* bine și e greșit.

**Prezice.** Cerem agentului: *„grupează angajații”*. Ce crezi că va include în calcul? Ce ar trebui să lase pe dinafară?

**Cere agentului** (mod executant, fără alte precizări):

> Mod executant: pe tabelul `general_data` din `lab-set2`, grupează angajații în 3 grupuri.

In [ ]:
# Cod de referință: ce face un agent grăbit cu o cerere vagă
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from seed.date import incarca
from sklearn.cluster import KMeans

df = incarca("lab-set2")[1]                 # tabelul general_data
brut = df.select_dtypes("number").dropna()  # toate coloanele numerice, așa cum sunt
print(brut.shape, "coloane:", list(brut.columns))

km = KMeans(n_clusters=3, n_init=10, random_state=42).fit(brut)
print("\nMărimea grupurilor:", sorted(np.bincount(km.labels_)))
brut.groupby(km.labels_)[["MonthlyIncome", "Age", "TotalWorkingYears", "EmployeeID"]].mean().round(0)

**Verifică.** Valori de control: `(4382, 16)` (28 de rânduri cu valori lipsă sunt scoase); grupuri de 566, 1.086 și 2.730 de angajați; veniturile medii pe grup ≈ 36.013, 84.727 și 167.442; vârsta medie ≈ 36-37 în toate trei.

**Capcană.** Cele trei „tipuri de angajați” sunt de fapt **trei trepte de venit**: vârsta, vechimea și toate celelalte coloane nu diferă între grupuri. Motivul: `MonthlyIncome` are valori de zeci de mii, celelalte de ordinul zecilor. Mai sunt amestecate un **ID** (`EmployeeID`, fără sens de măsurat) și două coloane **constante** (`EmployeeCount`, `StandardHours`). Acestea sunt regulile 1-2 din primer.

In [ ]:
# Pregătirea corectă: fără ID, fără coloane constante, fără valori lipsă
X = df.select_dtypes("number").drop(columns=["EmployeeID"])
X = X.loc[:, X.nunique() > 1].dropna()
print(X.shape)                                   # control: (4382, 13)
print("Coloane scoase:", sorted(set(df.select_dtypes("number").columns) - set(X.columns)))
print("Rânduri scoase (valori lipsă):", len(df) - len(X))

**Pentru client.** Numai după ce știm *ce* grupăm putem spune ceva util. De aici înainte lucrăm cu cele **13 coloane** și **4.382 de angajați** de mai sus.

## Scena 1. Distanța și scalarea

**Concept.** Algoritmii de grupare compară angajații prin **distanță**: doi angajați sunt „asemănători” dacă distanța dintre ei este mică. Distanța euclidiană între doi angajați $a$ și $b$ cu $p$ caracteristici:

$$d(a, b) = \sqrt{\sum_{j=1}^{p} (a_j - b_j)^2}$$

O coloană cu valori mari domină suma. **Standardizarea** (scorul $z$) pune toate coloanele pe aceeași scară:

$$z = \frac{x - \mu}{\sigma}$$

**Prezice.** Angajatul **A** are 51 de ani și un venit lunar de 131.160. **B** are 31 de ani și 135.000, **C** are 50 de ani și 160.000. Care dintre B și C ți se pare mai apropiat de A? Votați.

In [ ]:
# Cod de referință: distanța dintre A și B/C, înainte și după standardizare
A, B, C = np.array([51, 131160]), np.array([31, 135000]), np.array([50, 160000])
dist = lambda p, q: float(np.sqrt(((p - q) ** 2).sum()))

print("Fără scalare:  A-B =", round(dist(A, B)), "  A-C =", round(dist(A, C)))

sigma = X[["Age", "MonthlyIncome"]].std().to_numpy()   # abaterea standard a fiecărei coloane
print("Cu scalare:    A-B =", round(dist(A / sigma, B / sigma), 2), "  A-C =", round(dist(A / sigma, C / sigma), 2))

**Verifică.** Control: fără scalare A-B = 3.840 și A-C = 28.840 (B „mai aproape”); cu scalare A-B ≈ 2,19 și A-C ≈ 0,62 (C „mai aproape”).

**Capcană.** Aceiași trei oameni, **verdict inversat**. Fără scalare diferența de 20 de ani nu contează nimic față de diferența de 3.840 în venit. Dacă agentul nu standardizează și nu spune asta, grupurile tale sunt grupuri de venit (scena 0).

**Pentru client.** „Înainte de a grupa angajați, punem toate caracteristicile pe aceeași scară; altfel decide cea măsurată în cifre mari.”

## Scena 2. Corelația

**Concept.** Corelația Pearson între două coloane măsoară cât de bine urcă sau coboară împreună, între $-1$ și $1$:

$$r_{xy} = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sqrt{\sum_i (x_i - \bar x)^2}\,\sqrt{\sum_i (y_i - \bar y)^2}}$$

Nu depinde de scară (nu trebuie standardizat înainte) și măsoară doar relații *liniare*. Coloanele puternic corelate spun aproape același lucru: tocmai asta comprimă PCA în scena următoare.

**Prezice.** Dintre cele 13 coloane, care pereche ți se pare cea mai corelată? Dar `MonthlyIncome` cu `JobLevel` (nivelul în companie)?

**Cere agentului:**

> Pe `X` (cele 13 coloane), afișează matricea de corelații ca heatmap și cele cinci perechi cu cea mai mare corelație în valoare absolută.

In [ ]:
# Cod de referință
import seaborn as sns

corel = X.corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corel, cmap="coolwarm", center=0, vmin=-1, vmax=1, ax=ax, cbar_kws={"label": "corelație"})
ax.set_title("Vechimea (în firmă, cu managerul, de la ultima promovare) formează un bloc corelat")
plt.tight_layout()
plt.show()

perechi = corel.where(np.triu(np.ones(corel.shape, dtype=bool), k=1)).stack()
print(perechi.abs().sort_values(ascending=False).head(5).round(2))
print("\nMonthlyIncome - JobLevel:", round(corel.loc["MonthlyIncome", "JobLevel"], 2))

**Verifică.** Control: `MonthlyIncome`–`JobLevel` = 0,05; `YearsAtCompany`–`YearsWithCurrManager` = 0,77; `Age`–`TotalWorkingYears` = 0,68; `TotalWorkingYears`–`YearsAtCompany` = 0,63.

**Capcană.** Venitul și nivelul în companie ar trebui, într-o firmă reală, să meargă împreună. Dacă în date corelația este aproape de zero, asta spune ceva despre **date** (aici, un set de antrenament sintetic), nu despre firmă. Corelația ridicată nu înseamnă cauzalitate. `Education`, `JobLevel`, `StockOptionLevel` sunt ordinale codate cu numere: corelația lor e doar o aproximație.

**Pentru client.** „Vechimea în firmă, vechimea cu același manager și timpul de la ultima promovare se mișcă împreună; aduc puțină informație nouă una față de alta.”

## Scena 3. PCA: comprimăm 13 coloane în câteva componente

**Concept.** PCA rotește datele astfel încât prima axă (**componenta principală 1**) să surprindă cât mai multă variație, a doua cât mai multă din ce a rămas, și așa mai departe. Componentele sunt necorelate. Fiecare componentă $k$ explică o fracțiune din variația totală:

$$\text{varianță explicată}_k = \frac{\lambda_k}{\sum_{j=1}^{p} \lambda_j}$$

unde $\lambda_k$ sunt valorile proprii ale matricei de covarianță. **PCA nu folosește nicio țintă.** Păstrăm de obicei atâtea componente cât să ajungem la un prag, de exemplu 80%.

<img src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/truth.jpg" style="width: 320px">

*Fig. O proiecție care ascunde diferența dintre grupuri: la fel pot dispărea informații utile când reducem dimensiunea.*

**Prezice.** Avem 13 coloane, dintre care unele corelate. Câte componente crezi că trebuie pentru 80% din variație: 2, 4 sau 8?

**Cere agentului:**

> Standardizează `X`, aplică PCA și arată (1) varianța explicată de fiecare componentă și cumulat, într-un grafic, (2) numărul de componente pentru 80%, (3) cele mai mari încărcări (coeficienți) pentru primele două componente.

In [ ]:
# Cod de referință
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Z = StandardScaler().fit_transform(X)           # Z: cele 13 coloane standardizate
pca = PCA(random_state=42).fit(Z)
ve = pca.explained_variance_ratio_
cum = np.cumsum(ve)
n80 = int(np.argmax(cum >= 0.80)) + 1

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(1, 14), ve, label="pe componentă")
ax.plot(range(1, 14), cum, "o-", color="black", label="cumulat")
ax.axhline(0.80, ls="--", color="gray")
ax.set(xlabel="componenta", ylabel="varianță explicată",
       title=f"Sunt necesare {n80} componente din 13 pentru 80% din variație")
ax.legend()
plt.tight_layout()
plt.show()

print(f"PC1 {ve[0]:.1%}, PC2 {ve[1]:.1%}; primele două: {cum[1]:.1%}; componente pentru 80%: {n80}")
incarcari = pd.DataFrame(pca.components_[:2].T, index=X.columns, columns=["PC1", "PC2"])
incarcari.round(2).sort_values("PC1", key=abs, ascending=False)

**Verifică.** Control: PC1 22,9%, PC2 11,2%, primele două 34,1%; **8 componente** pentru 80% (81,1%). Încărcările mari pentru PC1: `YearsAtCompany` 0,51, `TotalWorkingYears` 0,48, `YearsWithCurrManager` 0,45, `YearsSinceLastPromotion` 0,42, `Age` 0,35. Pentru PC2: `NumCompaniesWorked` 0,65 și `Age` 0,50.

**Interpretare.** PC1 este un „indice de vechime”, PC2 un „indice de mobilitate” (câte firme a schimbat). Cu 13 coloane, 8 componente înseamnă că reducerea este **modestă**: datele nu sunt foarte redundante. Un agent care spune „2 componente sunt de ajuns” fără să arate 34% greșește.

In [ ]:
# Capcană: PCA fără standardizare
brut_pca = PCA(random_state=42).fit(X)
print("PC1 explică:", f"{brut_pca.explained_variance_ratio_[0]:.2%}")
pd.Series(brut_pca.components_[0], index=X.columns).round(3).sort_values(key=abs, ascending=False).head(3)

**Capcană.** Fără standardizare, PC1 „explică” ≈ 100% din variație, dar este doar `MonthlyIncome` (încărcare ≈ 1). Un rezultat „spectaculos” care nu înseamnă nimic, același mecanism ca în scenele 0 și 1.

**Pentru client.** „Cele 13 caracteristici se comprimă greu: ne trebuie 8 axe pentru 80% din informație. Cele mai importante două sunt vechimea și mobilitatea între firme.”

## Scena 4. k-means: câte tipuri de angajați?

**Concept.** k-means împarte angajații în $K$ grupuri; fiecare grup are un **centroid** (media lui). Algoritmul repetă: (1) fiecare angajat merge la cel mai apropiat centroid; (2) centroidul devine media noilor membri. Minimizează **inerția**, suma pătratelor distanțelor la centroid:

$$J = \sum_{j=1}^{K} \sum_{x_i \in C_j} \lVert x_i - \mu_j \rVert^2$$

Inerția scade mereu când crește $K$, deci nu poate alege singură $K$. Două ajutoare:
- **metoda cotului:** căutăm punctul de după care inerția scade lent;
- **coeficientul siluetă** (silhouette) pentru un angajat $i$, cu $a(i)$ = distanța medie la membrii propriului grup și $b(i)$ = distanța medie la cel mai apropiat alt grup:

$$s(i) = \frac{b(i) - a(i)}{\max\{a(i),\, b(i)\}} \in [-1, 1]$$

Media pe toți angajații se numește *scorul silhouette*. Reper uzual (Kaufman și Rousseeuw): peste 0,5 structură rezonabilă; 0,25-0,5 slabă; sub 0,25 practic fără structură.

**Prezice.** Credeți că angajații formează grupuri bine separate? Ce valoare a scorului silhouette vă așteptați?

**Cere agentului:**

> Pe `Z` rulează k-means pentru K de la 2 la 8 (`random_state=42`, `n_init=10`) și arată inerția și scorul silhouette pentru fiecare K, în două grafice.

In [ ]:
# Cod de referință
from sklearn.metrics import silhouette_score

rez = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Z)
    rez.append((k, km.inertia_, silhouette_score(Z, km.labels_)))
rez = pd.DataFrame(rez, columns=["K", "inertie", "silhouette"]).set_index("K")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
rez["inertie"].plot(marker="o", ax=a1, title="Inerția scade lin, fără „cot” clar")
rez["silhouette"].plot(marker="o", ax=a2, title="Silhouette: maxim la K=2, dar sub 0,25")
a2.axhline(0.25, ls="--", color="gray")
plt.tight_layout()
plt.show()
rez.round(3)

**Verifică.** Control: silhouette K=2 → 0,178; K=3 → 0,113; K=4 → 0,103; apoi ≈ 0,10. Inerția: 48.396 (K=2), 44.528 (K=3), 42.193 (K=4).

**Capcană.** „K cu silhouette maxim” alege K=2, dar 0,178 este **sub 0,25**: nu sunt grupuri naturale, ci o partiție a unui nor continuu. Un agent care raportează „K optim = 2” ca succes a ratat esențialul (regula 5 din primer). Alegem K și după **utilitate**: câte segmente poate folosi clientul și cât de bine se descriu? Vom lua **K = 3** și vom spune cinstit că structura este slabă.

## Scena 5. Segmentele și decizia

**Concept.** Un grup devine „tip de angajat” abia când îl **descriem prin coloanele originale** (nu prin numărul grupului sau prin componente) și îi dăm un nume pe care îl înțelege clientul.

**Prezice.** Cu K=3, ce trei profiluri vă așteptați? Credeți că unul dintre ele pleacă mult mai des?

**Cere agentului:**

> Rulează k-means cu K=3 pe `Z`. Arată, pentru fiecare grup, mărimea, mediile coloanelor originale din `X` și ponderea angajaților care au plecat (`Attrition == "Yes"`). Propune un nume pentru fiecare grup și arată grupurile pe primele două componente PCA.

In [ ]:
# Cod de referință
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(Z)
profil = X.groupby(km3.labels_).mean()
profil["n"] = np.bincount(km3.labels_)
profil["plecare_%"] = (df.loc[X.index, "Attrition"] == "Yes").groupby(km3.labels_).mean() * 100

# numele se dau după regula: cea mai mare vechime în firmă / cele mai multe firme / cel mai tânăr
nume = {profil["YearsAtCompany"].idxmax(): "Veterani",
        profil["NumCompaniesWorked"].idxmax(): "Săritori",
        profil["Age"].idxmin(): "Juniori"}
profil.index = [nume.get(i, f"grup {i}") for i in profil.index]
etichete = pd.Series(km3.labels_).map(nume).to_numpy()
profil[["n", "Age", "TotalWorkingYears", "YearsAtCompany", "NumCompaniesWorked",
        "YearsSinceLastPromotion", "MonthlyIncome", "plecare_%"]].round(1).sort_values("n")

In [ ]:
# Segmentele pe primele două componente
P = pca.transform(Z)[:, :2]
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for s in ["Veterani", "Săritori", "Juniori"]:
    ax.scatter(*P[etichete == s].T, s=6, alpha=0.5, label=s)
ax.set(xlabel="PC1 (indice de vechime)", ylabel="PC2 (indice de mobilitate)",
       title="Segmentele se întind pe cele două axe, dar se suprapun")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

**Verifică.** Control (K=3, `random_state=42`; ordinea etichetelor poate diferi, mărimile nu): grupuri de **1.009 / 1.380 / 1.993**.
- *Veterani* (1.009): vârstă ≈ 41, vechime în firmă ≈ 15 ani, ≈ 6,4 ani de la ultima promovare, plecare ≈ 10,7%.
- *Săritori* (1.380): ≈ 5 firme avute, vechime în firmă ≈ 4 ani, plecare ≈ 15,3%.
- *Juniori* (1.993): vârstă ≈ 31, vechime totală ≈ 6 ani, plecare ≈ 19,4%.

**Capcană.** Plecarea variază între 10,7% și 19,4%: segmentele **descriu** angajații, dar **nu prezic** cine pleacă; și nu am folosit `Attrition` la grupare. Venitul mediu e aproape același în toate trei (≈ 62-68 mii). A spune clientului „juniorii pleacă de două ori mai des” fără să arăți că structura e slabă (silhouette 0,11) ar fi o concluzie prea sigură. Segmentele nu rămân neapărat aceleași pe alte date; verificarea stabilității cu alt `random_state` sau alt eșantion este un pas bun.

**Ce propunem.** Veterani: stagnare (ani de la promovare) → plan de dezvoltare; Săritori: puțină vechime în firmă → integrare și mentorat; Juniori: pierderi mai mari → cercetăm motivele (ex. interviuri de ieșire) înainte să alegem intervenția.

## Ce îi spunem clientului

**Cere agentului:** „Scrie 3 propoziții pentru directorul de HR, cu cifrele din notebook și o limită a analizei.”

Rezumatul de mai jos este calculat din variabilele notebook-ului; compară-l cu ce a scris agentul: **fiecare cifră din rezumat trebuie să apară și în rezultatele tale**.

In [ ]:
# Rezumat construit din rezultatele de mai sus
pl = profil["plecare_%"]
print(f"1. Din {X.shape[1]} caracteristici numerice, {n80} componente explică 80% din variație: "
      f"datele se comprimă greu; primele două axe sunt vechimea și mobilitatea între firme.")
print(f"2. Cu K=3 găsim trei profiluri: {', '.join(profil.sort_values('n').index)} "
      f"({', '.join(str(int(v)) for v in profil.sort_values('n')['n'])} angajați).")
print(f"3. Structura este slabă (silhouette {rez.loc[3, 'silhouette']:.2f}) iar plecarea variază între "
      f"{pl.min():.1f}% și {pl.max():.1f}%: segmentele descriu angajații, dar nu prezic cine pleacă.")

---
# Referință

Definiții și detalii pentru acasă. Nu se parcurg la curs.

## R1. Standardizare, normalizare și alte scalări

| Metoda | Ce face | Când |
|---|---|---|
| Z-score (`StandardScaler`) | $z = (x-\mu)/\sigma$: media 0, abaterea standard 1 | cazul general: PCA, k-means, regresii regularizate |
| Min-Max (`MinMaxScaler`) | $x' = (x-x_{\min})/(x_{\max}-x_{\min})$: valori în $[0,1]$ | când vrem un interval fix; sensibil la valori extreme |
| Robustă (`RobustScaler`) | folosește mediana și cuartilele | când există valori aberante |
| Logaritm | reduce asimetria | valori pozitive cu coadă lungă (venituri, prețuri) |

*Standardizare* înseamnă de obicei scorul $z$; *normalizare* se folosește pentru Min-Max, dar în practică termenii se amestecă, așa că verificați ce a aplicat agentul.

**Regulă anti-scurgere (*data leakage*):** când modelul va fi evaluat pe date noi (modulele 4-6), împărțiți train/test **întâi**, apoi `fit` pe train și `transform` pe train și test. La PCA și clusterizare descriptivă, fără set de test, scalăm tot setul (fără țintă și fără ID-uri), ca la scena 3. Slide-uri: [extra-03-standardizare.pdf](extra-03-standardizare.pdf).

## R2. PCA pe scurt

1. Se standardizează datele; 2. se calculează matricea de covarianță; 3. se calculează vectorii și valorile proprii ale matricei; 4. vectorii proprii cu cele mai mari valori proprii sunt componentele principale; 5. se proiectează datele pe primele $k$ componente.

Componentele sunt combinații liniare ale coloanelor originale: $PC_1 = w_{11}x_1 + w_{12}x_2 + \dots + w_{1p}x_p$, unde coeficienții $w_{1j}$ sunt *încărcările*. Aceeași descompunere se obține și prin SVD. Limite: surprinde doar relații liniare; componentele sunt greu de interpretat; este sensibil la valori aberante; nu folosește ținta (o componentă mică poate fi tocmai cea care separă clasele).

<img src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/pca_procedure.png" style="width: 520px">

*Fig. PCA pe scurt (Credits [machinelearningcoban](https://machinelearningcoban.com/2017/06/15/pca/)).*

**Selecția față de extracția caracteristicilor.** Selecția păstrează un subset din coloanele originale (filtre, wrappers, metode încorporate); extracția creează coloane noi (PCA, LDA). Metodele precum t-SNE sau UMAP sunt pentru *vizualizare* în 2D, nu pentru a pregăti date pentru un model.

**Blestemul dimensionalității.** Cu cât sunt mai multe coloane, cu atât distanțele între puncte se aseamănă și modelele supraînvață mai ușor.

<img src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/curse_of_dim.jpg" style="width: 380px">

## R3. Indicele Davies-Bouldin

Pentru grupurile $C_i$ cu dispersia $s_i$ (distanța medie la centroid) și distanța dintre centroizi $d_{ij}$:

$$DB = \frac{1}{K}\sum_{i=1}^{K} \max_{j \ne i} \frac{s_i + s_j}{d_{ij}}$$

Valori mai mici sunt mai bune (grupuri compacte și îndepărtate). `sklearn.metrics.davies_bouldin_score`.

## R4. Alte metode de clusterizare

- **Ierarhică** (aglomerativă sau divizivă): arbore de îmbinări (*dendrogramă*), nu cere $K$ dinainte.
- **DBSCAN:** grupuri de formă oarecare, bazate pe densitate; marchează punctele izolate ca zgomot.
- **k-means:** presupune grupuri compacte, aproximativ sferice, de mărimi apropiate; funcționează doar pe coloane numerice. Pentru coloane categoriale: codificare one-hot (cu atenție la scară) sau metode dedicate (k-prototypes, distanța Gower).

<img src="https://raw.githubusercontent.com/sbildea/SEED/main/cursuri/03-invatare-nesupervizata/imagini/overview-of-clustering.png" style="width: 720px">

*Fig. Metode de clusterizare în scikit-learn.*

## R5. Exemple de utilizare

Segmentarea clienților (marketing), detecția anomaliilor (fraudă, securitate), gruparea documentelor pe teme, compresia imaginilor, gruparea pacienților cu simptome asemănătoare.

**Discuție.** Segmentarea după comportament online a fost folosită la influențarea alegătorilor (scandalul Cambridge Analytica). Ce date ar trebui să nu folosim niciodată într-o segmentare?

## R6. Resurse

- Documentația scikit-learn: [clusterizare](https://scikit-learn.org/stable/modules/clustering.html) și [PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca).
- [K-means vizual (video)](https://www.youtube.com/watch?v=R2e3Ls9H_fc)
- Cod complet, pe iris: [extra-03-pca-clusterizare](extra-03-pca-clusterizare.ipynb).

---
## Ce urmează

Am găsit structură în datele despre angajați: coloane care se mișcă împreună (PCA) și trei segmente (k-means), cu limitele lor. Cealaltă jumătate a modulului caută structură în **coșurile de cumpărături**: ce produse se cumpără împreună.

Continuăm cu [curs-03b-asociere-recomandari.ipynb](curs-03b-asociere-recomandari.ipynb). Puntea curs → seminar, cu ce ceri agentului și ce verifici, este la finalul lui.

## Jurnal de sesiune

Completează la sfârșit (sau cere agentului: *„închide scena”*) și păstrează-l în copia ta a notebook-ului. Dacă agentul rămâne fără tokeni, lipește primerul și acest jurnal unui alt agent și scrie „continuă de aici”.

- Modul / scena la care am ajuns:
- Agent folosit și data:
- Ce am decis și de ce:
- Valori de control verificate (da / nu / care):
- Greșeli găsite la agent:
- Întrebare deschisă:
- Pentru următorul agent (ce a rămas de făcut, într-o propoziție):